In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

import celltypist
from celltypist import models
from celltypist.models import Model

In [ ]:
# Nucseq rawdata
adata_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_raw.h5ad")

In [ ]:
adata_raw.obs['sample'].value_counts()

In [ ]:
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)
adata_raw.raw=adata_raw

In [ ]:
sc.pp.highly_variable_genes(adata_raw, flavor='seurat', n_top_genes=2000)
sc.pp.scale(adata_raw, max_value=10)
sc.tl.pca(adata_raw, svd_solver='arpack', n_comps=30)
print(adata_raw.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_raw, color='batch')

In [ ]:
# t-sne 
sc.tl.tsne(adata_raw, use_rep='X_pca', n_pcs=20)
# umap
sc.pp.neighbors(adata_raw, n_pcs=20, use_rep='X_pca')
sc.tl.umap(adata_raw)
sc.pl.umap(adata_raw, color=['batch'])

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(
    adata_raw, 
    color='batch',
    size=20,
    alpha=0.8,
    #legend_fontsize=12
    ax=ax  
)
plt.show()

In [ ]:
adata_raw.write("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_umap.h5ad")

In [ ]:
adata_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_umap.h5ad")

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_raw.obsm['X_pca'],adata_raw.obs, 'sample')

# Harmony 결과를 저장
adata_raw.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_raw, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_raw)
sc.pl.umap(adata_raw, color=['batch'])

In [ ]:
adata_raw.write("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_harmony.h5ad")

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_harmony.h5ad")
adata

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(
    adata, 
    color='batch',
    size=20,
    alpha=0.8,
    #legend_fontsize=12
    ax=ax  
)
plt.show()

In [ ]:
adata_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_raw.h5ad")
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)

In [ ]:
adata_raw.obs_names[adata_raw.obs_names.duplicated()]

In [ ]:
predictions_liver = celltypist.annotate(adata_raw, model = 'Healthy_Human_Liver.pkl', majority_voting = True)
adata_raw = predictions_liver.to_adata()

In [ ]:
adata.obs['celltypist_Liver']=adata_raw.obs['predicted_labels']
adata

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sc.pl.umap(
    adata, 
    color='celltypist_Liver',
    size=20,
    alpha=0.8,
    #legend_fontsize=12
    ax=ax  
)
plt.show()

In [ ]:
adata.write("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_harmony.h5ad")

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_harmony.h5ad")

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = adata.obsm['X_pca']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 20)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
sc.tl.leiden(adata, resolution=0.1, key_added="leiden")
sc.pl.umap(adata, color=['leiden'])

In [ ]:
sc.pl.umap(
    adata,
    color='leiden',
    frameon=False,
    size=10
)

In [ ]:
sc.tl.dendrogram(adata,groupby='leiden')
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata,
    var_names=top_genes,
    groupby="leiden",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=True,
    show=True,
    figsize=(10, 7)
)

In [ ]:
sc.pl.heatmap(
    adata,
    var_names=top_genes,
    groupby="leiden",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Nuc_total_leidenMarkers.csv")

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 80
groups = adata.obs['leiden'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata.obs[adata.obs['leiden'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata[selected_indices]

In [ ]:
sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="leiden",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False
)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = adata.obs['leiden']
batch_labels = adata.obs['batch']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(3, 7))  # color=... 도 추가 가능

# 그래프 설정
plt.title('Major cell lineages')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='Normal vs MASLD', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()



In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata.obs['batch']
cluster = adata.obs['leiden']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters, key=int):  # 클러스터 번호 순 정렬
    # 각 조건별 개수 계산
    donor_in = ((state == "normal") & (cluster == clus)).sum()
    donor_out = ((state == "normal") & (cluster != clus)).sum()
    masld_in = ((state == "MASLD") & (cluster == clus)).sum()
    masld_out = ((state == "MASLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [masld_in, masld_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    masld_frac = masld_in / (masld_in + masld_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "masld_fraction": masld_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values("p_value")
print(df_result)
# cluster 열을 숫자형으로 바꾼 뒤 정렬
df_result['cluster'] = df_result['cluster'].astype(int)
df_result = df_result.sort_values('cluster')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='normal')
plt.bar(x + width/2, df_result['masld_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['masld_fraction'][i])
        plt.text(x[i], max_height + 0.005, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
adata.write("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_harmony.h5ad")

In [ ]:
adata.obs

In [ ]:
sc.tl.dendrogram(adata,groupby='batch')
sc.tl.rank_genes_groups(adata, groupby='batch', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10)
rank_df_state = sc.get.rank_genes_groups_df(adata, group=None)
top_genes_state = rank_df_state.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()

In [ ]:
rank_df_state

In [ ]:
rank_df_state.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Nuc_total_stateMarkers.csv")

In [ ]:
sc.pl.umap(
    adata,
    color='leiden',
    frameon=False,
    size=10
)